In [1]:
import pandas as pd
import sys
sys.path.insert(0, "..")
from sqlalchemy import create_engine
from utils import fill_empty_rows, create_file_path, import_datasets_to_db

In [2]:
# Read the .csv file
df_nissan = pd.read_csv(create_file_path("nissanR.csv"))

In [3]:
# Print the head
df_nissan.head(2)

,ModType,FS1 Hardware,HITACHI PN,Hitachi Cache,NISSAN PN,Last Grp,Nissan Cache,EEPROM \nTYPE,23703 BLANK,23710 / 23703 PN,Example VIN,Year,Make & Model,Engine,Trans & Emissions,EPC Supression
0,NS-H20,H20-39FM0,BEM172-300 A1,J6,NEC009-075,??,??,NaN,NaN,23703-9FM0A,1N6AA0ECXEN504194,2014,Nissan Titan,5.6L VK56DE,"AT, FED",NaN
1,NS-H20,H20-39FM0,BEM172-300 A1,J6,NEC011-018,??,??,NaN,NaN,23703-9FM0A,5N1AA0NC3FN604105,2015,Nissan Armada,5.6L VK56DE,"AT, FED",NaN


In [4]:
# Get the index with empty rows
df_nissan_empty_rows_index = df_nissan[df_nissan['ModType'].isna() == True].index

In [5]:
# Drop the empty rows
df_nissan_drop_empty = df_nissan.drop(df_nissan_empty_rows_index)

In [6]:
# Drop the repeated columns
df_nissan_repeated_cols_index = df_nissan_drop_empty[df_nissan_drop_empty['ModType'] == 'ModType'].index

In [7]:
# Drop the repeated cols
df_nissan_repeated_cols_dropped = df_nissan_drop_empty.drop(df_nissan_repeated_cols_index)

In [8]:
# Remove rows where all cols have null (NaN) values,
# keeping only rows with at least one valid (non-null) entry.
df_nissan_rm_null_rows = df_nissan_repeated_cols_dropped[~df_nissan_repeated_cols_dropped.isnull().all(axis=1)]

In [9]:
# Keep only rows where more than one col has a non-null value.
# This removes rows with only a single non-null entry, ensuring each row has multiple valid data points.
df_nissan_rm_one_null_rows = df_nissan_rm_null_rows[df_nissan_rm_null_rows.notnull().sum(axis=1) != 1]

In [10]:
# Show cols
df_nissan_rm_one_null_rows.columns

Index(['ModType', 'FS1 Hardware', 'HITACHI PN', 'Hitachi Cache', 'NISSAN PN',
       'Last Grp', 'Nissan Cache', 'EEPROM \nTYPE', '23703 BLANK',
       '23710 / 23703 PN', 'Example VIN', 'Year', 'Make & Model', 'Engine',
       'Trans & Emissions', 'EPC Supression'],
      dtype='object')

# Cols to keep:
- FS1 Hardware 
- ModType
- HITACHI PN
- NISSAN PN 
- 23710 / 23703 PN

In [11]:
# List with cols to keep
cols_to_keep_list = [
        "FS1 Hardware",
        "ModType",
        "HITACHI PN",
        "NISSAN PN",
        "23710 / 23703 PN"
]

In [12]:
# Remove unnecessary columns
df_nissan_cols_to_keep = df_nissan_rm_one_null_rows[cols_to_keep_list]

In [13]:
# Dict o rename the cols
cols_to_keep_dict = {
    "FS1 Hardware": "hardware",
    "ModType": "pn1",
    "HITACHI PN": "pn2",
    "NISSAN PN": "pn3",
    "23710 / 23703 PN": "pn4"
}

In [14]:
# Create a df with needed cols
df_nissan_rn_cols = df_nissan_cols_to_keep.rename(columns=cols_to_keep_dict)

In [15]:
# Call the function to fill out null values
df_nissan_fill_nulls = fill_empty_rows(df_nissan_rn_cols)

In [16]:
# Show cols
df_nissan_fill_nulls.columns

Index(['hardware', 'pn1', 'pn2', 'pn3', 'pn4'], dtype='object')

In [17]:
# Check df info
df_nissan_fill_nulls.info()

<class 'pandas.core.frame.DataFrame'>
Index: 2747 entries, 0 to 2796
Data columns (total 5 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   hardware  2747 non-null   object
 1   pn1       2747 non-null   object
 2   pn2       2747 non-null   object
 3   pn3       2747 non-null   object
 4   pn4       2747 non-null   object
dtypes: object(5)
memory usage: 128.8+ KB


In [18]:
# Call the function to import the df cleaned to the postgreSQLdb
import_datasets_to_db(df_nissan_fill_nulls, 'part_numbers_nissan')

747